In [1]:
#import 
import pandas as pd 
import seaborn as sns 
import matplotlib.pyplot as plt 
import numpy as np


In [2]:
#load processed data 
df_particles=pd.read_csv('processed_data/cleaned_fine_particles_data.csv')
df_no2=pd.read_csv('processed_data/cleaned_nitrogen_dioxide_data.csv')
df_traffic=pd.read_csv('processed_data/cleaned_traffic_data.csv')

In [4]:
print("RAW TRAFFIC TYPE:", type(df_traffic['DateTime'].iloc[0]))
print("RAW PARTICLES TYPE:", type(df_particles['TimePeriod'].iloc[0]))

RAW TRAFFIC TYPE: <class 'str'>
RAW PARTICLES TYPE: <class 'str'>


In [3]:
#add text col of boro

# 1hot encoding cols 
cols = ['Borough_Manhattan', 'Borough_Brooklyn', 'Borough_Queens', 'Borough_Staten Island']

# add the text for the 4 defined (particles)
df_particles['Borough'] = df_particles[cols].idxmax(axis=1).str.replace('Borough_', '')

# add manhattan (particles)
df_particles.loc[df_particles[cols].sum(axis=1) == 0, 'Borough'] = 'Bronx'

#save
df_particles.to_csv('processed_data/cleaned_fine_particles_data.csv', index=False)

# add the text for the 4 defined (no2)
df_no2['Borough'] = df_no2[cols].idxmax(axis=1).str.replace('Borough_', '')

# add manhattan (no2)
df_no2.loc[df_no2[cols].sum(axis=1) == 0, 'Borough'] = 'Bronx'

#save 
df_no2.to_csv('processed_data/cleaned_nitrogen_dioxide_data.csv', index=False)

traffic_cols=['Boro_Manhattan', 'Boro_Brooklyn', 'Boro_Queens', 'Boro_Staten Island']

# add the text for the 4 defined (traffic)
df_traffic['Boro'] = df_traffic[traffic_cols].idxmax(axis=1).str.replace('Boro_', '')

# add manhattan (traffic)
df_traffic.loc[df_traffic[traffic_cols].sum(axis=1) == 0, 'Boro'] = 'Bronx'

#save 
df_traffic.to_csv('processed_data/cleaned_traffic_data.csv', index=False)

In [6]:
#sql stuff
import sqlite3
conn = sqlite3.connect('main.db')

#update the staten island col name + date
df_particles = df_particles.rename(columns={
    'Borough_Staten Island': 'Borough_Staten_Island', 
    'TimePeriod': 'Date'
})
df_no2 = df_no2.rename(columns={
    'Borough_Staten Island': 'Borough_Staten_Island',
    'TimePeriod': 'Date'
})
#update the names for traffic 
df_traffic = df_traffic.rename(columns={
    'Boro_Manhattan': 'Borough_Manhattan',
    'Boro_Brooklyn': 'Borough_Brooklyn',
    'Boro_Queens': 'Borough_Queens',
    'Boro_Staten Island': 'Borough_Staten_Island',
    'DateTime': 'Date',
    'Boro': 'Borough'
})

#cleaning the datetime cols 

#get the fulldatetime object 
df_traffic['Date'] = pd.to_datetime(df_traffic['Date'])
df_particles['Date'] = pd.to_datetime(df_particles['Date'])
df_no2['Date'] = pd.to_datetime(df_no2['Date'])

#add a season to traffic using the month 
df_traffic['Month'] = df_traffic['Date'].dt.month
rules = [
    df_traffic['Month'].isin([12, 1, 2]), # winter
    df_traffic['Month'].isin([6, 7, 8])   # summer
]
choices = ['Winter', 'Summer']
df_traffic['Season'] = np.select(rules, choices, default='Other')

# get the year 
df_traffic['Date'] = df_traffic['Date'].dt.year
df_particles['Date'] = df_particles['Date'].dt.year
df_no2['Date'] = df_no2['Date'].dt.year

#fixing structure 
base_cols = ['Date', 'Borough', 'Borough_Brooklyn', 'Borough_Manhattan', 'Borough_Queens', 'Borough_Staten_Island']

df_particles_new = df_particles.melt(
    id_vars=base_cols,
    value_vars=['Summer mean mcg/m3', 'Winter mean mcg/m3'],
    var_name='Season',
    value_name='Fine_Particles_Value'
)

df_no2_new = df_no2.melt(
    id_vars=base_cols,
    value_vars=['Summer mean ppb', 'Winter mean ppb'],
    var_name='Season',
    value_name='NO2_Value'
)

df_particles_new['Season'] = df_particles_new['Season'].str.replace(' mean mcg/m3', '')
df_no2_new['Season'] = df_no2_new['Season'].str.replace(' mean ppb', '')

#create tables

df_traffic.to_sql('traffic_volume', conn, if_exists='replace', index=False)
df_particles_new.to_sql('fine_particles', conn, if_exists='replace', index=False)
df_no2_new.to_sql('nitrogen_dioxide', conn, if_exists='replace', index=False)


#test 
print("testing...")
print("Traffic Columns:", df_traffic.columns.tolist())
print("Particles Columns:", df_particles.columns.tolist())
print("NO2 Columns:", df_no2.columns.tolist())
print("...testing")

#end of test

#master table 
master_query = """
CREATE TABLE master_data AS
SELECT 
    t.Date AS Year, 
    t.Borough, 
    p.Season, 
    t.Vol AS Traffic_Volume, 
    p.Fine_Particles_Value, 
    n.NO2_Value,
    t.Borough_Manhattan,
    t.Borough_Brooklyn,
    t.Borough_Queens,
    t.Borough_Staten_Island
FROM traffic_volume t
JOIN fine_particles p 
    ON t.Date = p.Date AND t.Borough = p.Borough AND t.Season = p.Season
JOIN nitrogen_dioxide n
    ON t.Date = n.Date AND t.Borough = n.Borough AND t.Season = n.Season;
"""

conn.execute("DROP TABLE IF EXISTS master_data") 
conn.execute(master_query)

testing...
Traffic Columns: ['RequestID', 'Vol', 'SegmentID', 'WktGeom', 'Direction', 'Date', 'Vol_log', 'Borough_Brooklyn', 'Borough_Manhattan', 'Borough_Queens', 'Borough_Staten_Island', 'Borough', 'Month', 'Season']
Particles Columns: ['Date', 'BoroID', 'Annual mean mcg/m3', 'Summer mean mcg/m3', 'Winter mean mcg/m3', 'Annual mean mcg/m3_log', 'Summer mean mcg/m3_log', 'Winter mean mcg/m3_log', 'Borough_Brooklyn', 'Borough_Manhattan', 'Borough_Queens', 'Borough_Staten_Island', 'Borough']
NO2 Columns: ['Date', 'BoroID', 'Annual mean ppb', 'Summer mean ppb', 'Winter mean ppb', 'Annual mean ppb_log', 'Summer mean ppb_log', 'Winter mean ppb_log', 'Borough_Brooklyn', 'Borough_Manhattan', 'Borough_Queens', 'Borough_Staten_Island', 'Borough']
...testing


In [10]:
check_df = pd.read_sql("SELECT * FROM master_data LIMIT 10", conn)
check_df.head()


# print("--- TRAFFIC SAMPLE ---")
# print(pd.read_sql("SELECT Date, Borough FROM traffic_volume LIMIT 1", conn).values)

# print("\n--- PARTICLES SAMPLE ---")
# print(pd.read_sql("SELECT Date, Borough, Season FROM fine_particles LIMIT 1", conn).values)

# print("\n--- NO2 SAMPLE ---")
# print(pd.read_sql("SELECT Date, Borough, Season FROM nitrogen_dioxide LIMIT 1", conn).values)

,Year,Borough,Season,Traffic_Volume,Fine_Particles_Value,NO2_Value,Borough_Manhattan,Borough_Brooklyn,Borough_Queens,Borough_Staten_Island
0,1970,Bronx,Winter,245,6.1,18.6,0.0,0.0,0.0,0.0
1,1970,Bronx,Winter,245,6.1,19.7,0.0,0.0,0.0,0.0
2,1970,Bronx,Winter,245,6.1,19.9,0.0,0.0,0.0,0.0
3,1970,Bronx,Winter,245,6.1,20.9,0.0,0.0,0.0,0.0
4,1970,Bronx,Winter,245,6.1,21.8,0.0,0.0,0.0,0.0


In [48]:
print("Traffic Date Type:", df_traffic['Date'].dtype)
print("Particles Date Type:", df_particles['Date'].dtype)
print("NO2 Date Type:", df_no2['Date'].dtype)

Traffic Date Type: int32
Particles Date Type: int32
NO2 Date Type: int32
